# Train RayGNN v3 with two-GPU DDP on Kaggle
Select Kaggle GPU T4 x2. Uses independent per-GPU loaders, synchronized gradients, FP16 mixed precision, and resumable optimizer checkpoints. Run the v3 correctness gates in the design document before a long training run. Increase LOCAL_BATCH only after measuring throughput and memory.


In [ ]:
import random
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU, then restart the session.'
GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT == 2, f'Expected two Kaggle GPUs, found {GPU_COUNT}.'
DEVICE = torch.device('cuda:0')
SEED = 17
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('PyTorch:', torch.__version__)
print('Training GPUs:', [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])


In [ ]:
# Clone or update the branch containing RayGNN and scored-FEN loader changes.
import subprocess
from pathlib import Path

REPO = Path('/kaggle/working/nnue-pytorch')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'raygnn',
                    'https://github.com/lualum/nnue-pytorch.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'raygnn'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'raygnn'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD'], check=True)
SOURCE_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', SOURCE_COMMIT)
import sys
loaded = any(name == package or name.startswith(package + '.')
             for name in sys.modules for package in ('raygnn', 'data_loader'))
if loaded and globals().get('_RAYGNN_LOADED_COMMIT') != SOURCE_COMMIT:
    raise RuntimeError('Restart the Kaggle kernel, then run the notebook from the top. '
                       'Python still has RayGNN or the native loader from an earlier checkout loaded.')
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'python-chess==0.31.4'], check=True)
subprocess.run(['cmake', '-S', str(REPO / 'data_loader/cpp'), '-B', str(REPO / 'build'),
                '-DCMAKE_BUILD_TYPE=Release', f'-DLIB_COPY_DIR={REPO}'], check=True)
subprocess.run(['cmake', '--build', str(REPO / 'build'), '-j4'], check=True)

import os, sys
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from raygnn import RayGNN, RayGNNConfig, RayGNNEvaluator, fens_to_batch
from data_loader import FenBatchProvider
from data_loader._native import FenBatch
assert RayGNN().layers[0].update[0].in_features == 310, 'The cloned branch needs the v3 RayGNN changes.'
assert hasattr(FenBatch, 'get_fens_and_scores'), 'The cloned branch needs the scored-FEN loader changes.'
_RAYGNN_LOADED_COMMIT = SOURCE_COMMIT


In [ ]:
# Read binpacks in place. A second file provides an independent validation stream.
binpacks = sorted(Path('/kaggle/input').rglob('*.binpack'), key=lambda p: p.stat().st_size, reverse=True)
if len(binpacks) < 2:
    raise FileNotFoundError('Attach separate training and validation .binpack files for v3.')
TRAIN_BINPACK = binpacks[0]
VAL_BINPACK = binpacks[1]
for path in binpacks:
    with path.open('rb') as handle:
        header = handle.read(4)
    if header != b'BINP':
        raise ValueError(f'Invalid binpack header: {path}')
    print(path, f'{path.stat().st_size / 2**30:.2f} GiB')
print('Train:', TRAIN_BINPACK)
print('Validation:', VAL_BINPACK)


In [ ]:
# The binpack score is side-to-move positive in Stockfish internal units.
# The loader's win-rate model uses 208 units per pawn, so convert to White-positive pawn units.
import chess
from torch.nn import functional as F

BATCH_SIZE = 512 * GPU_COUNT
SCREENING_POSITIONS = 5_000_000
STEPS_PER_INTERVAL = 256
POSITIONS_PER_INTERVAL = BATCH_SIZE * STEPS_PER_INTERVAL
INTERVALS = (SCREENING_POSITIONS + POSITIONS_PER_INTERVAL - 1) // POSITIONS_PER_INTERVAL
TOTAL_POSITIONS = INTERVALS * POSITIONS_PER_INTERVAL
SCHEDULE_INTERVALS = (20_000_000 + POSITIONS_PER_INTERVAL - 1) // POSITIONS_PER_INTERVAL
VALIDATION_STEPS = 100
LEARNING_RATE = 3e-4
RUN_DIR = Path('/kaggle/working/raygnn_v3_run_2gpu')
RUN_DIR.mkdir(parents=True, exist_ok=True)

def make_stream(path, cyclic):
    return FenBatchProvider(str(path), cyclic=cyclic, num_workers=2,
                            batch_size=BATCH_SIZE, include_scores=True)

def encode_scored_batch(fens, scores):
    rows = [(fen, score) for fen, score in zip(fens, scores)
            if abs(score) < 30000]  # omit mate-like labels
    if not rows:
        return None
    fens, scores = zip(*rows)
    batch = fens_to_batch(list(fens))
    stm = batch.side_to_move[:, 0]
    target = torch.tensor(scores, dtype=torch.float32) * stm / 208.0
    return batch, target[:, None].to(DEVICE)

@torch.inference_mode()
def validation_huber(network):
    if VAL_BINPACK is None:
        return None
    stream = make_stream(VAL_BINPACK, cyclic=False)
    network.eval()
    total, count = torch.zeros((), device=DEVICE), 0
    try:
        for _ in range(VALIDATION_STEPS):
            try:
                sample = encode_scored_batch(*next(stream))
            except StopIteration:
                break
            if sample is None:
                continue
            batch, target = sample
            prediction = network(batch.piece, batch.side_to_move, batch.castling, batch.en_passant)
            total += F.huber_loss(prediction, target, delta=1.0, reduction='sum')
            count += len(target)
    finally:
        del stream
    return (total / count).item() if count else None

print(f'{TOTAL_POSITIONS:,} sampled positions in {INTERVALS} intervals; '
      f'validate and save every {POSITIONS_PER_INTERVAL:,} positions')


In [ ]:
DDP_SOURCE = "import argparse, os, random, time\nfrom pathlib import Path\nfrom dataclasses import asdict\nimport torch, torch.distributed as dist\nfrom torch.nn import functional as F\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom raygnn import RayGNN, RayGNNConfig, fens_to_batch\nfrom raygnn.encoding import swap_colors_reflect_ranks\nfrom data_loader import FenBatchProvider\n\np=argparse.ArgumentParser()\nfor name in ('train','val','out'): p.add_argument('--'+name,required=True)\np.add_argument('--positions',type=int,default=5_000_000)\np.add_argument('--local-batch',type=int,default=512)\np.add_argument('--steps-per-interval',type=int,default=256)\np.add_argument('--loader-workers',type=int,default=4)\np.add_argument('--validation-steps',type=int,default=100)\np.add_argument('--lr',type=float,default=3e-4)\np.add_argument('--resume',default='')\na=p.parse_args()\nrank=int(os.environ['RANK']); local=int(os.environ['LOCAL_RANK']); world=int(os.environ['WORLD_SIZE'])\nassert world==2 and torch.cuda.device_count()==2, 'Expected exactly two GPUs'\ntorch.cuda.set_device(local); device=torch.device('cuda',local)\ndist.init_process_group('nccl')\nrandom.seed(17+rank); torch.manual_seed(17); torch.cuda.manual_seed_all(17)\ntorch.backends.cudnn.benchmark=True\ntry: torch.set_float32_matmul_precision('high')\nexcept Exception: pass\nout=Path(a.out); out.mkdir(parents=True,exist_ok=True)\nconfig=RayGNNConfig()\nmodel=RayGNN(config).to(device)\ndecay=[p for name,p in model.named_parameters() if p.requires_grad and p.ndim>1 and 'piece_square' not in name]\nno_decay=[p for name,p in model.named_parameters() if p.requires_grad and (p.ndim<=1 or 'piece_square' in name)]\noptimizer=torch.optim.AdamW([{'params':decay,'weight_decay':1e-4},{'params':no_decay,'weight_decay':0.0}],lr=a.lr)\nper_interval=a.local_batch*world*a.steps_per_interval\nintervals=(a.positions+per_interval-1)//per_interval\nschedule_intervals=max(intervals,(20_000_000+per_interval-1)//per_interval)\ntotal_steps=schedule_intervals*a.steps_per_interval\ndef lr_factor(step):\n    if step<200:return (step+1)/200\n    progress=min(1.,(step-200)/max(1,total_steps-200))\n    return 0.1+0.9*(1.+__import__('math').cos(__import__('math').pi*progress))/2\nscheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_factor)\nscaler=torch.amp.GradScaler('cuda')\nstart_interval=0; best=float('inf')\nif a.resume:\n    checkpoint=torch.load(a.resume,map_location='cpu',weights_only=False)\n    model.load_state_dict(checkpoint['model'])\n    if 'optimizer' in checkpoint: optimizer.load_state_dict(checkpoint['optimizer'])\n    if 'scheduler' in checkpoint: scheduler.load_state_dict(checkpoint['scheduler'])\n    if 'scaler' in checkpoint: scaler.load_state_dict(checkpoint['scaler'])\n    start_interval=int(checkpoint.get('screening_interval',0))\n    best=float(checkpoint.get('best',float('inf')))\nnetwork=DDP(model,device_ids=[local],output_device=local,broadcast_buffers=False,gradient_as_bucket_view=True)\n\ndef stream(path,cyclic):\n    return FenBatchProvider(str(path),cyclic=cyclic,num_workers=a.loader_workers,\n                            batch_size=a.local_batch,include_scores=True)\ndef encode(fens,scores):\n    rows=[(fen,score) for fen,score in zip(fens,scores) if abs(score)<30000]\n    if not rows:return None\n    fens,scores=zip(*rows)\n    batch=fens_to_batch(list(fens))\n    stm=batch.side_to_move[:,0]\n    target=torch.tensor(scores,dtype=torch.float32)*stm/208.0\n    return batch,target[:,None].to(device,non_blocking=True)\ndef reduce_stats(total,count):\n    t=torch.stack((total,count));dist.all_reduce(t,op=dist.ReduceOp.SUM)\n    return (t[0]/t[1]).item() if t[1].item()>0 else None\n@torch.inference_mode()\ndef validate():\n    if not a.val:return None\n    network.eval(); total=torch.zeros((),device=device);count=torch.zeros((),device=device)\n    val_stream=stream(a.val,False)\n    try:\n        # Independent loader streams per rank; validation metrics aggregate both ranks.\n        for _ in range(a.validation_steps):\n            try: sample=encode(*next(val_stream))\n            except StopIteration:break\n            if sample is None:continue\n            batch,target=sample\n            with torch.autocast('cuda',dtype=torch.float16):\n                pred=network(batch.piece,batch.side_to_move,batch.castling,batch.en_passant)\n                loss=F.huber_loss(pred.float(),target,delta=1.,reduction='sum')\n            total+=loss;count+=len(target)\n    finally:del val_stream\n    return reduce_stats(total,count)\ntrain_stream=stream(a.train,True)\nstart=time.perf_counter()\nif rank==0:print('DDP GPUs:',world,'local batch:',a.local_batch,'global batch:',a.local_batch*world,'positions/interval:',per_interval,flush=True)\ntry:\n    for interval in range(start_interval,intervals):\n        network.train(); total=torch.zeros((),device=device);count=torch.zeros((),device=device)\n        load_time=0.;tick=time.perf_counter()\n        for step in range(a.steps_per_interval):\n            t=time.perf_counter();sample=encode(*next(train_stream));load_time+=time.perf_counter()-t\n            if sample is None:continue\n            batch,target=sample\n            mask=torch.rand(len(target))<0.5\n            if mask.any():\n                flipped=swap_colors_reflect_ranks(batch)\n                batch.piece[mask]=flipped.piece[mask]\n                batch.side_to_move[mask]=flipped.side_to_move[mask]\n                batch.castling[mask]=flipped.castling[mask]\n                batch.en_passant[mask]=flipped.en_passant[mask]\n                target[mask.to(device)]=-target[mask.to(device)]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast('cuda',dtype=torch.float16):\n                prediction=network(batch.piece,batch.side_to_move,batch.castling,batch.en_passant)\n                loss=F.huber_loss(prediction.float(),target,delta=1.)\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            scale_before=scaler.get_scale()\n            scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=scale_before:scheduler.step()\n            total+=loss.detach()*len(target);count+=len(target)\n        torch.cuda.synchronize();seconds=time.perf_counter()-tick\n        train_loss=reduce_stats(total,count)\n        val_loss=validate()\n        positions=(interval+1)*per_interval\n        perf=torch.tensor([seconds,load_time,torch.cuda.max_memory_allocated()/2**30],device=device)\n        perf_all=[torch.zeros_like(perf) for _ in range(world)]\n        dist.all_gather(perf_all,perf)\n        if rank==0:\n            elapsed=(time.perf_counter()-start)/3600\n            print(f'{positions:,}/{intervals*per_interval:,} positions: train Huber={train_loss:.4f}, '\n                  f'validation Huber={val_loss}, GPU times/load/GiB={[v.tolist() for v in perf_all]}, '\n                  f'elapsed={elapsed:.2f}h',flush=True)\n            ckpt={'model':model.state_dict(),'architecture_version':'v3','relation_classes':['direct','xray','context'],'config':asdict(config),'optimizer':optimizer.state_dict(),\n                  'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),\n                  'positions_seen':positions,'screening_interval':interval+1,'best':best,\n                  'target':'White-positive pawn units = binpack score * side_to_move / 208',\n                  'input_schema':'piece[64], side_to_move=+1/-1, castling=WK/WQ/BK/BQ, en_passant=0..64',\n                  'edge_feature_order':['relative_delta','is_knight','blocker_count_one_hot','first_blocker_piece_one_hot','first_blocker_delta','blocker_present','geometry_matches','direct_attack_defense','source_occupied','destination_occupied','pawn_forward_geometry'],\n                  'perspective':'White-positive','train_binpack':a.train,'validation_huber':val_loss}\n            selection=val_loss if val_loss is not None else train_loss\n            if selection<best:\n                best=selection;ckpt['best']=best\n                torch.save(ckpt,out/'best.pt')\n            torch.save(ckpt,out/'last.pt')\nfinally:\n    del train_stream\n    dist.destroy_process_group()\n"

# True two-process, two-GPU DDP. Each process loads its own batches and owns one GPU.
import os, subprocess, sys
from pathlib import Path
DDP_SCRIPT = REPO / 'kaggle_raygnn_ddp.py'
DDP_SCRIPT.write_text(DDP_SOURCE)
LOCAL_BATCH = 512  # per GPU; try 384/512 only if memory and throughput permit
LOADER_WORKERS_PER_GPU = 4
RESUME = ''  # e.g. str(RUN_DIR / 'last.pt'); only resumes DDP checkpoints with optimizer state
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
           str(DDP_SCRIPT), '--train', str(TRAIN_BINPACK), '--val', str(VAL_BINPACK or ''),
           '--out', str(RUN_DIR), '--positions', str(SCREENING_POSITIONS),
           '--local-batch', str(LOCAL_BATCH), '--loader-workers', str(LOADER_WORKERS_PER_GPU)]
if RESUME: command += ['--resume', RESUME]
env = os.environ.copy()
env.update(OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONUNBUFFERED='1',
           NCCL_DEBUG='WARN', TORCH_NCCL_ASYNC_ERROR_HANDLING='1')
subprocess.run(command, cwd=str(REPO), env=env, check=True)


In [ ]:
# Confirm that the saved model can be loaded by the engine adapter.
evaluator = RayGNNEvaluator.from_checkpoint(RUN_DIR / 'best.pt', DEVICE)
print('Starting-position White-positive centipawns:', evaluator.evaluate_cp([chess.Board()]).item())
for path in sorted(RUN_DIR.glob('*.pt')):
    print(path, f'{path.stat().st_size / 2**20:.1f} MiB')
